# ปฏิบัติการรายวิชา อจวพ ๓๐๒ ปฏิบัติการสารสนเทศด้านสุขภาพ
## สัปดาห์ที่ ๒: การเชื่อมโยงข้อมูลและการรวมกลุ่มข้อมูลสุขภาพ (Data Linking & Aggregation)
---

## 🛠️ Setup สำหรับ Windows ด้วย scoop + uv (Self-study, รันออฟไลน์ได้)

> **ทำตามได้บน Windows 10/11 โดยไม่ต้องพึ่งเซิร์ฟเวอร์มหาวิทยาลัย** — ใช้ PowerShell (ไม่ใช่ CMD)

**1. ติดตั้ง scoop (ครั้งเดียว):**
```powershell
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser
Invoke-RestMethod -Uri https://get.scoop.sh | Invoke-Expression
scoop --version   # ทดสอบว่าติดตั้งสำเร็จ
```
คู่มือทางการ: [scoop.sh](https://scoop.sh) | [Scoop docs (GitHub)](https://github.com/ScoopInstaller/Scoop)

**2. ติดตั้ง git + uv ผ่าน scoop:**
```powershell
scoop install git uv
uv --version
```
เอกสารทางการ: [uv installation](https://docs.astral.sh/uv/getting-started/installation/) | [uv — managing projects](https://docs.astral.sh/uv/guides/projects/)

**3. สร้างโปรเจกต์และติดตั้งไลบรารี (ในโฟลเดอร์ health-informatics ของคุณ):**
```powershell
D:
mkdir health-informatics; cd health-informatics
uv init
uv add jupyterlab pandas faker tietai-synthea
# หรือถ้ามีโปรเจกต์อยู่แล้ว: uv add tietai-synthea
```

**4. เปิด JupyterLab แบบ reproducible:**
```powershell
uv run jupyter lab
# เปิดเบราว์เซอร์ที่ http://localhost:8888 แล้วเปิดไฟล์ .ipynb นี้
```

**หมายเหตุ PySynthea (แทน Synthea Java เดิม):**
- PyPI: [tietai-synthea](https://pypi.org/project/tietai-synthea/) — import ชื่อ `synthea`, คำสั่ง CLI `synthea`
- GitHub: [TIET-AI/tietai-synthea](https://github.com/TIET-AI/tietai-synthea) (Quick Start, API, disease modules)
- Paper: [PySynthea (PDF)](https://tiet.ai/pdf/pysynthea-paper.pdf) | Blog: [tiet.ai/blog/py-synthea](https://tiet.ai/blog/py-synthea/)
- ไม่ต้องติดตั้ง Java/JVM — รันด้วย `uv run synthea -p 10` หรือ Python API `from synthea import Generator, GeneratorOptions`
- ข้อมูลสังเคราะห์จะส่งออกเป็น FHIR R4 JSON Bundle — สเปกทางการ: [HL7 FHIR R4](https://www.hl7.org/fhir/) | [FHIR Patient](https://www.hl7.org/fhir/patient.html) | [FHIR Bundle](https://www.hl7.org/fhir/bundle.html)

**อ้างอิง pandas / Jupyter ที่ใช้ในแล็บนี้:**
- [pandas docs](https://pandas.pydata.org/docs/) | [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) | [pandas I/O](https://pandas.pydata.org/pandas-docs/stable/reference/io.html) | [Indexing & selecting](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [JupyterLab docs](https://jupyterlab.readthedocs.io/en/latest/)

> ทุกเซลล์ในโน้ตบุ๊กนี้รันด้วย `uv run` จึงล็อกเวอร์ชันผ่าน `uv.lock` — ส่งโปรเจกต์ให้อาจารย์แล้วรันซ้ำได้ผลเดิม (reproducible)


### วัตถุประสงค์การเรียนรู้ (Learning Objectives)
1. เข้าใจหลักการและสามารถเขียนโค้ดเพื่อ **เชื่อมโยงตารางข้อมูล (Data Merging/Linking)** ด้วยไลบรารี `pandas` ได้อย่างถูกต้อง (CLO2)
2. สามารถประยุกต์ใช้ฟังก์ชัน **groupby** และ **agg** ในการจัดกลุ่มข้อมูลและสรุปผลทางสถิติสะสม (Data Aggregation) เชิงระบบสุขภาพได้ (CLO2)
3. เข้าใจหลักการคัดกรองข้อมูลระดับกลุ่มเสมือนเงื่อนไข **HAVING** ใน SQL (CLO2)
4. ฝึกวิเคราะห์และดึงข้อมูลจากโครงสร้างกึ่งมีโครงสร้างตามมาตรฐานสากล **HL7 FHIR JSON** (CLO1, CLO4)

### การนำเข้าไลบรารีที่จำเป็น

In [ ]:
import pandas as pd
import json
print("นำเข้าไลบรารีสำเร็จ!")

---
### ส่วนที่ ๑: การเชื่อมโยงข้อมูลจากหลายตาราง (Data Linking / Merging)

ในระบบสารสนเทศสุขภาพจริง ข้อมูลเวชระเบียนของผู้ป่วย บันทึกการจ่ายยา และข้อมูลพิกัดภูมิศาสตร์สถานพยาบาลมักจะเก็บแยกคนละตารางเพื่อลดความซ้ำซ้อนของข้อมูล (Normalization) ปฏิบัติการนี้จะทดลองเชื่อมโยงตาราง:
1. **`prescriptions.csv`**: ตารางบันทึกข้อมูลการจ่ายยาสามัญสะสม
2. **`practices_registry.csv`**: ตารางทะเบียนภูมิศาสตร์ที่ตั้งของคลินิกต่างๆ

เราจะโหลดตารางข้อมูลทั้งสองเข้ามาตรวจสอบมิติก่อนทำการเชื่อมโยงข้อมูล

In [ ]:
# 1. โหลดข้อมูล
df_prescriptions = pd.read_csv('prescriptions.csv')
df_practices = pd.read_csv('practices_registry.csv')

# 2. แสดงมิติข้อมูลและตัวอย่าง
print("ขนาดตารางสั่งใช้ยา:", df_prescriptions.shape)
print("ขนาดตารางทะเบียนคลินิก:", df_practices.shape)
display(df_prescriptions.head(3))
display(df_practices.head(3))

#### การเชื่อมตารางด้วยวิธี Left Join และระบุ Suffixes
ในงานด้านสาธารณสุข เรามักนิยมใช้การเชื่อมแบบ `Left Join` เพื่อรักษาฐานระเบียนหลักฝั่งซ้าย (ในที่นี้คือรายการสั่งใช้ยาทั้งหมด) ป้องกันข้อมูลผู้ป่วยตกหล่นเชิงระบบสถิติ และการใช้ `suffixes` เพื่อแยกคอลัมน์ชื่อซ้ำ

In [ ]:
df_linked = pd.merge(
    df_prescriptions, 
    df_practices, 
    on='practice_code', 
    how='left'
)

print("ขนาดหลังเชื่อมโยงข้อมูล:", df_linked.shape)
display(df_linked.head(5))

---
### ส่วนที่ ๒: การรวมกลุ่มข้อมูลและการหาผลลัพธ์สถิติสะสม (Data Aggregation & Group By)

การประเมินระบบสุขภาพระดับนโยบาย มักต้องการตัวชี้วัดภาพรวมระดับองค์กรหรือกลุ่มประชากร (Cohort KPIs) เราจะวิเคราะห์ข้อมูลด้วยกระบวนการ **Split-Apply-Combine** เพื่อคำนวณสถิติรายคลินิก เช่น:
- **ผลรวมงบประมาณรายจ่ายค่ายาของแต่ละคลินิก** (`sum` ของคอลัมน์ `actual_cost`)
- **ราคายาเฉลี่ยต่อระเบียนสั่งจ่าย** (`mean` ของคอลัมน์ `actual_cost`)
- **จำนวนใบสั่งจ่ายยาทั้งหมด** (`count` ของรายการ)

In [ ]:
practice_summary = df_linked.groupby('practice_name').agg(
    total_spend=('actual_cost', 'sum'),
    average_drug_price=('actual_cost', 'mean'),
    total_prescriptions=('patient_id', 'count')
).reset_index()

display(practice_summary)

---
### ส่วนที่ ๓: การกรองผลลัพธ์ในระดับกลุ่ม (Having Concept)

หากนักศึกษาต้องการคัดเลือกเฉพาะสถานพยาบาลที่มียอดความหนาแน่นของผู้ป่วยสูงหรือใช้จ่ายงบประมาณเกินเกณฑ์ เราจะไม่สามารถกรองในขั้นตอน `WHERE` ดั้งเดิมได้ เนื่องจากข้อมูลยังไม่ได้รับการรวมกลุ่มสถิติ เราจึงต้องเขียนตัวกรองแบบ `Having` บน DataFrame สรุป

In [ ]:
# กรองเฉพาะสถานพยาบาลที่มียอดสั่งจ่ายยาสะสมมากกว่า 50 ใบสั่งขึ้นไป
high_volume_practices = practice_summary[
    practice_summary['total_prescriptions'] > 50
].sort_values(by='total_spend', ascending=False)

display(high_volume_practices)

---
### ส่วนที่ ๔: การจัดการและแปลงข้อมูลมาตรฐานเวชระเบียนสากล (HL7 FHIR JSON)

ในคาบเรียนนี้นักศึกษาได้เรียนรู้แนวคิดมาตรฐานส่งผ่านข้อมูลทางการแพทย์สากลอย่าง **HL7 FHIR (Fast Healthcare Interoperability Resources)** ซึ่งบันทึกในรูปแบบ Nested JSON วันนี้เราจะฝึกทักษะการใช้ฟังก์ชัน `pd.json_normalize` เพื่อคลี่คุณลักษณะของผู้ป่วยออกมาวิเคราะห์เป็นตารางสถิติ

In [ ]:
# 1. เปิดอ่านข้อมูล FHIR Bundle
with open('patient_fhir_demo.json', encoding='utf-8') as f:
    fhir_data = json.load(f)

# 2. คลี่ข้อมูลผู้ป่วยที่อยู่ภายใต้ 'entry' ออกมาเป็นตาราง
df_fhir_patients = pd.json_normalize(
    fhir_data, 
    record_path=['entry']
)

display(df_fhir_patients)

---
### ส่วนที่ ๕: โจทย์ปฏิบัติการการบ้านชุดที่ ๒ (Homework 2)

**รายละเอียดข้อกำหนดการส่งงาน (คะแนนเต็ม ๒๐ คะแนน):**
ให้นักศึกษาเขียนโปรแกรมเชื่อมโยงข้อมูลและวิเคราะห์ตัวชี้วัดเพื่อตอบโจทย์ทางการแพทย์ดังต่อไปนี้:
1. ทำการเชื่อมตารางแบบ **Left Join** ระหว่างตารางสั่งจ่ายยา `prescriptions.csv` และทะเบียนคลินิก `practices_registry.csv` ผ่านรหัส `practice_code` 
2. ทำการจัดกลุ่มข้อมูล (**Group By**) เพื่อหาค่าเฉลี่ยสถิติราคายารวมสะสมแยกตาม **ประเภทพื้นที่ที่ตั้งคลินิก (area_type)** และหาค่าเฉลี่ยระดับ **ความดันโลหิตบนเฉลี่ย (average systolic_bp)**
3. ทำการกรองผลลัพธ์คัดเลือกเฉพาะพื้นที่ที่มี **ยอดความดันโลหิตบนเฉลี่ยสูงเกินกว่า 120 mmHg** เท่านั้น
4. บันทึกผลลัพธ์เป็นไฟล์ CSV ชื่อ **`student_id_homework2.csv`** โดยไม่ต้องฝังดัชนีระเบียบลงไป และอัปโหลดไฟล์รหัสปฏิบัติการส่งตามกำหนดการ

In [ ]:
# === เขียนโค้ดส่งการบ้าน 2 ด้านล่างนี้ ===

